# GeoCalib-Align Colab Notebook

End-to-end Colab workflow for free, open-source fine-tuning and evaluation.

## Section 0: Workspace Setup

This notebook assumes the repository files are already available in Colab, either by cloning the repository into `/content/geocalib_align` or by uploading the project folder.

In [ ]:
from pathlib import Path
import os

REPO_URL = os.environ.get("GEOCALIB_ALIGN_REPO_URL", "https://github.com/elom354/geocalib_align.git")
REPO_DIR = Path("/content/GEOCALIB_ALIGN")

if REPO_DIR.exists():
    print("Repo already present at", REPO_DIR)
else:
    !git clone "$REPO_URL" "$REPO_DIR"

%cd /content/GEOCALIB_ALIGN
print("Repo ready at", REPO_DIR)

Cloning into '/content/GEOCALIB_ALIG'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 45 (delta 4), reused 45 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 40.82 KiB | 746.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/GEOCALIB_ALIG
Repo ready at /content/GEOCALIB_ALIG


## Section 1: Environment Setup

Dependencies are installed in a Colab-friendly way without downgrading core system packages such as NumPy, SciPy, Torch, or `fsspec`. This avoids binary incompatibility errors and reduces resolver conflicts with the preloaded Colab stack.

In [2]:
%%time
from pathlib import Path
import importlib
import os
import subprocess
import sys

sentinel = Path('/tmp/geocalib_align_colab_deps_v2')
if not sentinel.exists():
    required = {
        'transformers': 'transformers==4.46.3',
        'peft': 'peft==0.13.2',
        'bitsandbytes': 'bitsandbytes==0.45.0',
        'accelerate': 'accelerate==0.34.2',
        'trl': 'trl==0.10.1',
        'bert_score': 'bert-score==0.3.13',
        'kaleido': 'kaleido==0.2.1',
    }
    missing = [pkg for module_name, pkg in required.items() if importlib.util.find_spec(module_name) is None]
    if missing:
        print({'installing': missing})
        cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', *missing]
        subprocess.run(cmd, check=True)
        sentinel.write_text('installed')
        raise SystemExit(
            'New packages were installed successfully. Stop here, use Runtime > Restart session, then rerun the notebook from the top.'
        )
    else:
        sentinel.write_text('installed')
        print('Required packages already available in this runtime.')
else:
    print('Dependency layer already installed for this runtime.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.3/297.3 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import os
from getpass import getpass
import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
gpu_vram = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else 0
print({'gpu_available': torch.cuda.is_available(), 'gpu_name': gpu_name, 'gpu_vram_gb': gpu_vram})
os.environ['HF_TOKEN'] = getpass('Hugging Face token (optional, leave blank for public models): ')

{'gpu_available': True, 'gpu_name': 'Tesla T4', 'gpu_vram_gb': 14.56}
Hugging Face token (optional, leave blank for public models): ··········


## Section 2: Data Loading

GeoBench, EarthSE, and GeoSignal are downloaded automatically by the repository scripts using verified dataset IDs.

In [4]:
%%time
!python data/download_data.py
!python data/prepare_geosignal.py
!python data/build_physgeo_eval.py

import json
import matplotlib.pyplot as plt
from pathlib import Path

train_records = json.loads(Path('data/processed/geosignal_train.json').read_text())
for sample in train_records[:3]:
    print(sample)

lengths = [len((row['instruction'] + ' ' + row['input'] + ' ' + row['output']).split()) for row in train_records]
plt.hist(lengths, bins=30)
plt.title('GeoSignal Token Length Distribution')
plt.xlabel('Approximate word count')
plt.ylabel('Frequency')
plt.show()

INFO: Attempting to load dataset: Deng2023/GeoBench
INFO: Attempting to load dataset: Deng2023/GeoBench
INFO: Attempting to load dataset: GeoX-Lab/GeoBench
Traceback (most recent call last):
  File "/content/GEOCALIB_ALIG/data/download_data.py", line 89, in <module>
    main()
  File "/content/GEOCALIB_ALIG/data/download_data.py", line 65, in main
    geobench = _download_with_fallback(args.geobench_id, ["Deng2023/GeoBench", "GeoX-Lab/GeoBench"])
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/GEOCALIB_ALIG/data/download_data.py", line 53, in _download_with_fallback
    raise RuntimeError("Unable to download dataset. Errors:\n" + "\n".join(errors))
RuntimeError: Unable to download dataset. Errors:
Deng2023/GeoBench: Dataset 'Deng2023/GeoBench' doesn't exist on the Hub or cannot be accessed
Deng2023/GeoBench: Dataset 'Deng2023/GeoBench' doesn't exist on the Hub or cannot be accessed
GeoX-Lab/GeoBench: Dataset 'GeoX-La

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/geosignal_train.json'

## Section 3: Model Loading

A public non-gated model is used by default so the notebook stays free and does not require special access approval.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=quant_config, device_map='auto')
print(model.__class__.__name__)
print('Trainable parameters before LoRA:', sum(p.numel() for p in model.parameters() if p.requires_grad))

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Section 4: Fine-Tuning (Standard LoRA)

In [ ]:
%%time
!python finetune/train_lora_standard.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/lora_standard --config config/experiments.yaml

## Section 5: Fine-Tuning (Selective LoRA)

In [ ]:
%%time
!python finetune/train_lora_selective.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/lora_selective --config config/experiments.yaml

## Section 6: Fine-Tuning (LoRA + Replay)

In [ ]:
%%time
!python finetune/train_lora_replay.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/lora_replay --config config/experiments.yaml

## Section 7: Fine-Tuning (Mix-CPT)

In [ ]:
%%time
!python finetune/train_mix_cpt.py --model_id Qwen/Qwen2.5-1.5B-Instruct --output_dir results/colab_demo/mix_cpt --config config/experiments.yaml

## Section 8: Evaluation

The free evaluation stack uses local open-source models only. PhysGeo evaluation is skipped automatically until the annotation template has been filled with real expert-reviewed examples.

In [ ]:
%%time
!python evaluate/eval_closed_tasks.py --model_path results/colab_demo/lora_standard --model_name Qwen2.5-1.5B --strategy lora_std
!python evaluate/eval_open_tasks.py --model_path results/colab_demo/lora_standard --model_name Qwen2.5-1.5B --strategy lora_std --judge_model Qwen/Qwen2.5-1.5B-Instruct

import json
import subprocess
import sys
from pathlib import Path
import pandas as pd

template = json.loads(Path('data/physgeo_eval_template.json').read_text())
if template.get('examples'):
    subprocess.run([
        sys.executable,
        'evaluate/eval_physgeo.py',
        '--model_path',
        'results/colab_demo/lora_standard',
        '--model_name',
        'Qwen2.5-1.5B',
        '--strategy',
        'lora_std',
        '--judge_model',
        'Qwen/Qwen2.5-1.5B-Instruct',
    ], check=True)
    subprocess.run([sys.executable, 'evaluate/aggregate_results.py'], check=True)
    display(pd.read_csv('results/summary.csv').head())
else:
    print('PhysGeo template is empty by design. Add expert-reviewed examples to data/physgeo_eval_template.json, then rerun PhysGeo evaluation and aggregation.')
    open_results = json.loads(Path('results/Qwen2.5-1.5B/lora_std/open_results.json').read_text())
    closed_results = json.loads(Path('results/Qwen2.5-1.5B/lora_std/closed_results.json').read_text())
    print({'closed_overall': closed_results['overall_accuracy'], 'prompt_alignment': open_results['prompt_alignment'], 'correctness': open_results['correctness'], 'answer_relevance': open_results['answer_relevance']})

## Section 9: Figure Generation

Figures are generated only when `results/summary.csv` exists, which requires closed, open, and PhysGeo evaluation outputs.

In [ ]:
%%time
from pathlib import Path
import subprocess
import sys
from IPython.display import Image, display

summary_path = Path('results/summary.csv')
if summary_path.exists():
    subprocess.run([sys.executable, 'figures/generate_all_figures.py', '--data', str(summary_path)], check=True)
    figure_names = [
        'fig1_tradeoff_scatter',
        'fig2_radar_chart',
        'fig3_heatmap',
        'fig4_bar_strategies',
        'fig5_physgeo_breakdown',
        'fig6_cost_performance',
        'fig7_alignment_delta',
        'fig8_leaderboard',
    ]
    for figure_name in figure_names:
        display(Image(filename=f'figures/{figure_name}.png'))
else:
    print('Skipping figure generation because results/summary.csv does not exist yet.')

## Section 10: Export

In [ ]:
%%time
from pathlib import Path
from google.colab import files

summary_path = Path('results/summary.csv')
if summary_path.exists():
    !zip -r geocalib_align_figures.zip figures/*.pdf figures/*.png results/summary.csv
    files.download('results/summary.csv')
    files.download('geocalib_align_figures.zip')
else:
    print('Skipping export because results/summary.csv does not exist yet.')